In [5]:
import pandas as pd
import requests
import io
import gzip

# 1. COMPREHENSIVE MAPPING: Your list to ISCO-08 4-Digit Codes
# These mappings represent the closest international standard for each role.
profession_to_isco = {
    'Accountant': '2411', 'Actor': '2655', 'Actuary': '2120', 'Administrative Assistant': '3341',
    'Administrator': '1211', 'Air Traffic Controller': '3154', 'Animal Trainer': '5164', 'Anthropologist': '2632',
    'Appraiser': '3315', 'Archaeologist': '2632', 'Architect': '2161', 'Archivist': '2621',
    'Art Director': '2166', 'Artist': '2651', 'Astronaut': '2111', 'Astronomer': '2111',
    'Athlete': '3421', 'Audio Technician': '3521', 'Auditor': '2411', 'Automotive Designer': '2144',
    'Baker': '7512', 'Banker': '2412', 'Bankruptcy Specialist': '2412', 'Barber': '5141',
    'Barista': '5120', 'Bartender': '5132', 'Basketball player': '3421', 'Biologist': '2131',
    'Biomedical Engineer': '2149', 'Blacksmith': '7221', 'Bodyguard': '5414', 'Bounty Hunter': '5419',
    'Boxer': '3421', 'Brand Manager': '1221', 'Brewer': '7515', 'Bricklayer': '7112',
    'Broker': '3311', 'Builder': '7111', 'Butcher': '7511', 'CEO': '1120',
    'Carer': '5322', 'Carpenter': '7115', 'Cartographer': '2165', 'Cashier': '5230',
    'Chef': '5120', 'Chemical Engineer': '2145', 'Chemist': '2113', 'Chiropractor': '2269',
    'Civil Engineer': '2142', 'Claims Adjuster': '3315', 'Cleaner': '9112', 'Clerk': '4110',
    'Coach': '3422', 'Comedian': '2655', 'Compliance Officer': '2422', 'Composer': '2652',
    'Conservation Officer': '2133', 'Construction Worker': '9313', 'Copywriter': '2641', 'Court Reporter': '4120',
    'Crime Scene Investigator': '3119', 'Customer Support Specialist': '4222', 'DJ': '2652', 'Dancer': '2653',
    'Data Scientist': '2511', 'Database Administrator': '2521', 'Debt Counselor': '3313', 'Dentist': '2261',
    'Detective': '3355', 'Development Officer': '1221', 'Dietitian': '2265', 'Director': '1120',
    'Doctor': '2211', 'Dog Walker': '5164', 'Draughtsperson': '3118', 'Driver': '8322',
    'Economist': '2633', 'Editor': '2641', 'Electrician': '7411', 'Emergency Management Specialist': '1219',
    'Entrepreneur': '1120', 'Environmental Engineer': '2143', 'Ergonomist': '2149', 'Estate Planner': '2412',
    'Event Coordinator': '3332', 'Executive Assistant': '3341', 'Exterminator': '9213', 'Facilities Manager': '1439',
    'Farmer': '6111', 'Fashion Designer': '2163', 'Firefighter': '5411', 'Fishmonger': '5221',
    'Flight Attendant': '5111', 'Florist': '5221', 'Football player': '3421', 'Forklift Operator': '8344',
    'Gardener': '6113', 'Geologist': '2114', 'Graphic Designer': '2166', 'Grocer': '5221',
    'Hair dresser': '5141', 'Handyperson': '9622', 'Health Inspector': '3257', 'Historian': '2633',
    'Hotel Concierge': '5113', 'Hotel Manager': '1411', 'Human Resources Specialist': '2423', 'IT Support Specialist': '3512',
    'Illustrator': '2651', 'Industrial Designer': '2163', 'Insurance Underwriter': '3312', 'Janitor': '9112',
    'Jeweller': '7313', 'Journalist': '2642', 'Judge': '2612', 'Lawyer': '2611',
    'Librarian': '2622', 'Lifeguard': '5419', 'Loan Officer': '3312', 'Logger': '6210',
    'Logistics Manager': '1324', 'Magician': '2659', 'Makeup Artist': '5142', 'Marine Biologist': '2131',
    'Marketing Manager': '1221', 'Masseur': '3255', 'Mathematician': '2120', 'Mayor': '1111',
    'Mechanic': '7231', 'Meteorologist': '2112', 'Midwife': '2222', 'Miner': '8111',
    'Model': '5241', 'Musician': '2652', 'News Reader': '2642', 'Nurse': '2221',
    'Nutritionist': '2265', 'Oceanographer': '2114', 'Office Assistant': '4110', 'Operations Manager': '1324',
    'Optician': '3254', 'Painter': '7131', 'Paralegal': '3411', 'Paramedic': '2240',
    'Park Ranger': '3143', 'Payroll Specialist': '4313', 'Personal Trainer': '3423', 'Pharmacist': '2262',
    'Photographer': '3431', 'Physicist': '2111', 'Pilot': '3153', 'Plumber': '7126',
    'Police Officer': '5412', 'Politician': '1111', 'Postal Worker': '4412', 'Priest': '2637',
    'Procurement Officer': '3323', 'Professor': '2310', 'Property Manager': '1219', 'Psychologist': '2634',
    'Quality Assurance Inspector': '3139', 'Real Estate Agent': '3334', 'Receptionist': '4226', 'Researcher': '2111',
    'Roofer': '7121', 'Safety Inspector': '3257', 'Sailor': '8350', 'Salesperson': '5223',
    'Scientist': '2111', 'Security Officer': '5414', 'Shopkeeper': '5221', 'Singer': '2652',
    'Skier': '3421', 'Social Worker': '2635', 'Software Engineer': '2512', 'Soldier': '0110',
    'Sound Engineer': '3521', 'Statistician': '2120', 'Street Vendor': '9520', 'Surfer': '3421',
    'Surgeon': '2212', 'Swimmer': '3421', 'Tailor': '7531', 'Tattoo Artist': '3431',
    'Teacher': '2330', 'Technician': '3119', 'Tennis Player': '3421', 'Therapist': '2269',
    'Translator': '2643', 'Umpire': '3422', 'Urban Planner': '2164', 'Usher': '5414',
    'Veterinarian': '2250', 'Videographer': '3431', 'Waiter': '5131', 'Waste Collection Worker': '9611',
    'Welder': '7212', 'Wholesaler': '1420', 'Writer': '2641', 'Zoologist': '2131'
}

def download_and_process_gender_data():
    # Corrected Bulk URL: Files are in 'indicator/' and require the frequency suffix '_A'
    bulk_url = "https://ilostat.ilo.org/data/bulk/indicator/EMP_TEMP_SEX_OCU_NB_A.csv.gz"
    
    headers = {'User-Agent': 'Mozilla/5.0'}

    print("Step 1: Downloading global dataset...")
    response = requests.get(bulk_url, headers=headers)
    
    if response.status_code != 200:
        return f"Error: Received status {response.status_code}. The URL may have changed again."

    print("Step 2: Decompressing and loading data...")
    with gzip.open(io.BytesIO(response.content), 'rt') as f:
        # Load only necessary columns to save memory
        cols = ['ref_area.label', 'classif1', 'sex', 'time', 'obs_value']
        df = pd.read_csv(f, usecols=cols)

    print("Step 3: Filtering for your job list...")
    # Convert ISCO codes to the ILO format (e.g., '2411' -> 'OCU_ISCO08_2411')
    target_codes = [f"OCU_ISCO08_{code}" for code in set(profession_to_isco.values())]
    df = df[df['classif1'].isin(target_codes)]

    print("Step 4: Pivoting and calculating gender share...")
    # Pivot so Male and Female values are side-by-side
    pivot_df = df.pivot_table(
        index=['ref_area.label', 'classif1', 'time'],
        columns='sex',
        values='obs_value'
    ).reset_index()

    # Calculate Female Share
    if 'SEX_F' in pivot_df.columns and 'SEX_T' in pivot_df.columns:
        pivot_df['female_share_pct'] = (pivot_df['SEX_F'] / pivot_df['SEX_T']) * 100
        
        # Add a column for the original profession name
        inv_map = {}
        for k, v in profession_to_isco.items():
            inv_map.setdefault(f"OCU_ISCO08_{v}", []).append(k)
        
        pivot_df['Job_Names'] = pivot_df['classif1'].map(inv_map)
        
        # Get the latest year for each country/job combo
        final_df = pivot_df.sort_values('time').groupby(['ref_area.label', 'classif1']).last()
        
        return final_df[['female_share_pct', 'Job_Names']]
    else:
        return "Critical gender columns (SEX_F or SEX_T) missing in data."

# Run the script
final_results = download_and_process_gender_data()
print(final_results) #.head(20))

Step 1: Downloading global dataset...
Error: Received status 404. The URL may have changed again.


In [11]:
import pandas as pd
import requests
import io

def fetch_ilo_gender_data(isco_code):
    # Base URL for the production API
    base_url = "https://sdmx.ilo.org/rest/data" #[cite: 431]
    
    # Dataflow for Employment by Occupation and Sex
    dataflow = "ILO,DF_EMP_TEMP_SEX_OCU_NB" #[cite: 588, 649]
    
    # Key structure: REF_AREA.FREQ.MEASURE.SEX.AGE.OCCUPATION
    # Use wildcards (.) for Area, Freq, Measure, and Age.
    # Specify 'SEX_T+SEX_F' to get both Total and Female counts.
    key = f"....SEX_T+SEX_F..OCU_ISCO08_{isco_code}" #[cite: 470, 656, 657]
    
    # URL construction
    url = f"{base_url}/{dataflow}/{key}/"#[cite: 462]
    
    # Request parameters for CSV format and latest observations
    params = {
        "format": "csv",          # Directly request CSV [cite: 511, 713]
        "lastNObservations": 1    # Get only the most recent data point [cite: 477, 702]
    }
    
    headers = {'User-Agent': 'Mozilla/5.0'} # Essential to avoid 403 errors
    
    try:
        response = requests.get(url, params=params, headers=headers)
        if response.status_code == 200:
            return pd.read_csv(io.StringIO(response.text))
        else:
            return f"Error: {response.status_code}"# [cite: 758]
    except Exception as e:
        return f"Request failed: {e}"

# Example: Fetch data for Accountants (ISCO 2411)
df_accountant = fetch_ilo_gender_data("2411")
print(df_accountant) #.head())

Error: 404


In [12]:
import requests

url = (
    "https://sdmx.ilo.org/rest/data/"
    "ILO,DF_EMP_TEMP_SEX_OCU_NB/"
    "MLT.A.EMP_TEMP.SEX_F+SEX_T.OCU_ISCO08_2512"
)

params = {
    "format": "jsondata",
    "detail": "dataonly"
}

r = requests.get(url, params=params)
r.raise_for_status()

data = r.json()


HTTPError: 404 Client Error: Not Found for url: https://sdmx.ilo.org/rest/data/ILO,DF_EMP_TEMP_SEX_OCU_NB/MLT.A.EMP_TEMP.SEX_F+SEX_T.OCU_ISCO08_2512?format=jsondata&detail=dataonly

In [14]:
import requests

url = (
    "https://sdmx.ilo.org/rest/data/"
    "ILO,DF_EMP_TEMP_SEX_OCU_NB/"
    "MLT.A.NB.SEX_T.OCU_ISCO08_1"
)

params = {
    "format": "jsondata",
    "detail": "dataonly",
    "lastNObservations": 1
}

r = requests.get(url, params=params)
r.raise_for_status()

data = r.json()


HTTPError: 404 Client Error: Not Found for url: https://sdmx.ilo.org/rest/data/ILO,DF_EMP_TEMP_SEX_OCU_NB/MLT.A.NB.SEX_T.OCU_ISCO08_1?format=jsondata&detail=dataonly&lastNObservations=1